# Preview the first TOC text features

This notebook reads two local files:

- `data/raw/books.parquet`: the 9,034 saved book rows, including `raw_toc_json` and the existing `toc_entries`.
- `data/processed/toc_features.parquet`: one new row per raw TOC item, created by `bookpath.toc`.

If the processed file does not exist, run `uv run python -m bookpath.toc` from the repository root first. This notebook only reads files.

We use *Never Split the Difference* (`parent_asin = "0062407805"`) throughout the main preview. `base_text` is the selected chapter text; `feature_text` adds the chapter subtitle when it contributes additional words. This is a candidate text feature, not an embedding or a demonstrated improvement in recommendations.

## 1. Load the raw books and the processed TOC rows

Objective: Read the two saved files and record the raw file checksum before inspection.

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from bookpath.local_data import read_books_from_parquet

project_root = Path.cwd()
while not (project_root / "src/bookpath").is_dir():
    if project_root == project_root.parent:
        raise FileNotFoundError("Open this notebook inside the Bookpath repository.")
    project_root = project_root.parent

raw_path = project_root / "data/raw/books.parquet"
feature_path = project_root / "data/processed/toc_features.parquet"
if not feature_path.is_file():
    raise FileNotFoundError("Run uv run python -m bookpath.toc from the repository root first.")


def file_checksum(path):
    with path.open("rb") as saved_file:
        return hashlib.file_digest(saved_file, "sha256").hexdigest()


checksum_before = file_checksum(raw_path)
books = read_books_from_parquet(raw_path)
toc_features = pd.read_parquet(feature_path)
print(f"Raw book rows: {len(books):,}")
print(f"Processed TOC rows: {len(toc_features):,}")
print("Feature version:", toc_features["feature_version"].unique().tolist())

## 2. Compare the same first chapter in three places

Objective: Display the first chapter from the raw book's `raw_toc_json`, its existing `toc_entries`, and the new processed feature row. `raw_position = 1` identifies the first original TOC item; it does not depend on how many earlier items had usable text.

In [ ]:
example_asin = "0062407805"
selected_book = books.loc[books["parent_asin"].eq(example_asin)].iloc[0]
book_features = toc_features.loc[toc_features["parent_asin"].eq(example_asin)]
first_feature = book_features.loc[book_features["raw_position"].eq(1)].iloc[0]
raw_entry = json.loads(selected_book["raw_toc_json"])[0]
existing_entry = selected_book["toc_entries"][0]

print("Book:", selected_book["title"])
print("\n1. First item from the raw_toc_json column:")
print(json.dumps(raw_entry, ensure_ascii=False, indent=2))
print("\n2. First item from the existing toc_entries column:")
print(json.dumps(existing_entry, ensure_ascii=False, indent=2))
print("\n3. New feature_text in the processed file:")
print(first_feature["feature_text"])

Objective: See exactly which fields supplied the new text. The chapter label, level, and page stay in separate metadata columns. Book title, labels, author credits, and page numbers are not automatically joined into `feature_text`.

In [ ]:
comparison = pd.DataFrame({
    "field": ["existing toc_entries text", "base_text", "chapter_subtitle", "feature_text", "subtitle_added"],
    "value": [
        existing_entry["text"],
        first_feature["base_text"],
        first_feature["chapter_subtitle"],
        first_feature["feature_text"],
        first_feature["subtitle_added"],
    ],
})
with pd.option_context("display.max_colwidth", None):
    display(comparison)
    display(first_feature[[
        "parent_asin", "ol_edition_key", "raw_position", "text_source",
        "label", "level", "page", "entry_class",
    ]].rename("value").to_frame())

## 3. Read every proposed chapter text for this book

Objective: Read the 14 entries for the same book in their original order. Change `example_asin` in section 2 and rerun the following cells to inspect another book.

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(book_features[[
        "raw_position", "feature_text", "subtitle_added", "missing_text",
    ]].set_index("raw_position"))

## 4. Review flagged entries across all books

Objective: Count and inspect entries with no usable feature text, only digits, or repeated text within a book. All these rows remain in the processed file. Repeat comparison ignores case and repeated whitespace; it includes every occurrence in a repeated group.

In [ ]:
review_columns = ["missing_text", "numeric_only_text", "repeated_text"]
print("Entries with a chapter subtitle added:", int(toc_features["subtitle_added"].sum()))
display(toc_features[review_columns].sum().rename("entries").to_frame())

for flag in review_columns:
    print(f"Examples where {flag} is True:")
    display(toc_features.loc[toc_features[flag], [
        "parent_asin", "raw_position", "base_text", "feature_text",
    ]].head(5))

## 5. Verify the existing chapter text and raw item counts

Objective: Check each book separately: its nonempty `base_text` values should reproduce the existing `toc_entries` text in order, and the processed file should keep every raw item's position. These checks do not establish that the book's source TOC is complete.

In [ ]:
base_text_by_book = (
    toc_features.loc[toc_features["base_text"].notna()]
    .groupby("parent_asin")["base_text"].agg(list)
)
raw_positions_by_book = toc_features.groupby("parent_asin")["raw_position"].agg(list)
validation_rows = []
for book in books.itertuples():
    expected_text = [entry["text"] for entry in book.toc_entries]
    raw_count = len(json.loads(book.raw_toc_json))
    validation_rows.append({
        "parent_asin": book.parent_asin,
        "existing_text_matches": base_text_by_book.get(book.parent_asin, []) == expected_text,
        "raw_positions_preserved": raw_positions_by_book.get(book.parent_asin, []) == list(range(1, raw_count + 1)),
    })

validation = pd.DataFrame(validation_rows)
check_columns = ["existing_text_matches", "raw_positions_preserved"]
display(validation[check_columns].sum().rename("matching_books").to_frame())
assert validation[check_columns].all().all(), "Inspect books that failed the comparison."

## 6. Confirm which raw file produced these features

Objective: Check the saved source checksum, unique book/item identifiers, and the raw file's before/after checksum.

In [ ]:
assert toc_features["source_sha256"].eq(checksum_before).all()
assert not toc_features.duplicated(["parent_asin", "raw_position"]).any()
checksum_after = file_checksum(raw_path)
assert checksum_after == checksum_before, "The raw Parquet changed during inspection."
print(f"Source checksum: {checksum_before}")
print("All comparisons passed; raw file unchanged.")

The reusable code lives in [src/bookpath/toc.py](../src/bookpath/toc.py). It trims surrounding whitespace, chooses main text, and adds a chapter subtitle once. The substring check compares whole word sequences, ignoring case and punctuation; it is not a semantic similarity model.

For now, retain the review flags and source metadata. No entries are automatically deleted, hierarchy is not reconstructed, Amazon description TOCs are not merged, and no embeddings are computed. Define retrieval evaluation before deciding whether the fuller text helps recommendations.